In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import glob 
import os

In [ ]:
image_fp='./images'

In [ ]:
image_names=[os.path.basename(file) for file in glob.glob(os.path.join(image_fp,'*.jpg'))]

In [ ]:
image_names

In [ ]:
len(image_names)

In [ ]:
labels=[' '.join(name.split('_')[:-1:]) for name in image_names]

In [ ]:
labels


In [ ]:
def label_encode(label):
    if label == 'Abyssinian':
        return 0
    elif label == 'Bengal':
        return 1
    elif label == 'Birman':
        return 2
    elif label == 'Bombay':
        return 3
    elif label == 'British Shorthair':
        return 4
    elif label == 'Egyptian Mau':
        return 5
    elif label == 'Persian':
        return 6
    elif label == 'Ragdoll':
        return 7
    elif label == 'Russian Blue':
        return 8
    elif label == 'Siamese':
        return 9
    elif label == 'Sphynx':
        return 10
    elif label == 'american bulldog':
        return 11
    elif label == 'american pit bull terrier':
        return 12
    elif label == 'basset hound':
        return 13
    elif label == 'beagle':
        return 14
    elif label == 'boxer':
        return 15
    elif label == 'chihuahua':
        return 16
    elif label == 'english cocker spaniel':
        return 17
    elif label == 'english setter':
        return 18
    elif label == 'german shorthaired':
        return 19
    elif label == 'great pyrenees':
        return 20
    elif label == 'havanese':
        return 21
    elif label == 'japanese chin':
        return 22
    elif label == 'keeshond':
        return 23
    elif label == 'leonberger':
        return 24
    elif label == 'Maine Coon':
        return 25
    elif label == 'miniature pinscher':
        return 26
    elif label == 'newfoundland':
        return 27
    elif label == 'pomeranian':
        return 28
    elif label == 'pug':
        return 29
    elif label == 'saint bernard':
        return 30
    elif label == 'samoyed':
        return 31
    elif label == 'scottish terrier':
        return 32
    elif label == 'shiba inu':
        return 33
    elif label == 'staffordshire bull terrier':
        return 34
    elif label == 'wheaten terrier':
        return 35
    elif label == 'yorkshire terrier':
        return 36

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array,load_img

In [ ]:

features=[]
labels=[]
IMAGE_SIZE= (224, 224)

for name in image_names:
    label=' '.join(name.split('_')[:-1:])
    label_encoded=label_encode(label)

    if(label_encoded!=None):
        img=load_img(os.path.join(image_fp,name))
        img=tf.image.resize_with_pad(img_to_array(img,dtype='uint8'),*IMAGE_SIZE).numpy().astype('uint8')

        

        features.append(img)
        labels.append(label_encoded)

                   

In [ ]:
features

In [ ]:
labels

In [ ]:
features_array=np.array(features)
labels_array=np.array(labels)

In [ ]:
labels_one_hot=pd.get_dummies(labels_array).astype(int)

In [ ]:
labels_one_hot

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
#train-60% validate-20% test-20%
X_train, X_test, y_train, y_test = train_test_split(features_array,labels_one_hot,test_size=0.2,random_state=30)

In [ ]:
#80%
X_train,X_val,y_train,y_val=train_test_split(features_array,labels_one_hot,test_size=0.25,random_state=3)

In [ ]:
from tensorflow.keras import layers,Input,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as pp_i
from tensorflow.keras.layers import RandomFlip, RandomRotation, Dense, Dropout
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

In [ ]:
data_augementation=Sequential([RandomFlip('horizontal_and_vertical'),RandomRotation(0.2)])
prediction_layers = Dense(37, activation='softmax')

In [ ]:
resnet_model=ResNet50(include_top=False,weights='imagenet',pooling='avg')
resnet_model.trainable=False
preprocess_input=pp_i

In [ ]:
input=Input(shape=(224,224,3))
x=data_augementation(input)
x=preprocess_input(x)
x=resnet_model(x,training=False)
x=Dropout(0.2)(x)
output=prediction_layers(x)
model=Model(input,output)

In [ ]:
model.compile(optimizer=Adam(),loss=CategoricalCrossentropy(),metrics=['accuracy'])
model_history=model.fit(x=X_train,y=y_train,validation_data=(X_val, y_val),epochs=3)

In [ ]:
acc = model_history.history['accuracy']
val_acc = model_history.history['val_accuracy']

loss = model_history.history['loss']
val_loss = model_history.history['val_loss']

In [ ]:
epochs_range = range(len(acc))

plt.figure(figsize=(15, 8))

# Accuracy graph
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Loss graph
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

In [ ]:
model.evaluate(X_test,y_test)

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
y_pred
